# Nearest-Atom and H-bond Distance Analysis: Ground-Truth Waters

Three questions, all answered on **real ≥3-H-bond GT waters**:

| # | Question | Method |
|---|---|---|
| Q1 | For each rank k, what atom type is the k-th nearest atom to each water? | KD-tree over all heavy atoms; stacked bar by identity |
| Q2 | For each atom type, what is the distribution of minimum distances to a water O? | Per-type KD-tree; histogram + global min |
| Q3 | What is the distribution of distances to actual H-bond partners? | N/O/F atoms in [2.5, 3.5] Å; histogram split by partner element |

All helper functions live in [`nearest_atom_helpers.py`](nearest_atom_helpers.py).

**To swap the validation set**: change `PDB_LIST_FILE` in the Setup cell.

---
## Part 0 — Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

_here = Path('.').resolve()
sys.path.insert(0, str(_here))
sys.path.insert(0, str(_here.parent))  # imputation/

from nearest_atom_helpers import (
    # Tier 2: atom array building
    build_all_atom_info, build_hbond_candidate_info,
    # Tier 3: Q1
    get_k_nearest_identities,
    # Tier 4: Q2
    compute_atomtype_nearest_dists,
    # Tier 5: Q3
    compute_hbond_partner_distances,
    # Tier 6: water coords
    extract_solvent_coords,
    # Tier 7: runners
    analyze_one_pdb, collect_results,
    # Tier 8: aggregation
    aggregate_k_nearest, aggregate_dist_dict, aggregate_hbond_contacts,
    # Tier 9: plotting
    set_paper_style,
    plot_k_nearest_stacked_bar,
    plot_atomtype_min_dist_distributions,
    plot_hbond_distance_histogram,
    print_atomtype_min_table,
    HBOND_MIN_DIST, HBOND_MAX_DIST, NPZ_ROOT,
)
from boltzgen.data.data import Structure
from boltzgen.data import const
from basic_helpers import resolve_npz_path
from gloria_hbond_helpers import gloria_remove_weak_solvents

set_paper_style()

In [ ]:
# ── PDB ID list — swap this path to use a different validation set ─────────────
PDB_LIST_FILE = Path('../easy.txt')  # <-- change to '../hard.txt' or another file

pdb_ids = PDB_LIST_FILE.read_text().split()
print(f'{len(pdb_ids)} PDB IDs: {pdb_ids[:6]} ...')

In [ ]:
# ── Analysis hyperparameters ──────────────────────────────────────────────────
MIN_HBONDS = 3    # minimum H-bonds for a real water to be included
K_MAX      = 10   # how many nearest atoms to retrieve (Q1)

DEMO_PDB = pdb_ids[0]
print(f'Demo PDB: {DEMO_PDB}')

---
## Part 1 — Single-PDB walkthrough

Step through the analysis for one PDB entry so each intermediate is inspectable.

### 1a. Load and filter the GT structure

In [ ]:
npz_path = resolve_npz_path(DEMO_PDB, NPZ_ROOT)
gt_raw   = Structure.load(npz_path)
gt_raw   = gt_raw.to_one_solvent_per_chain(gt_raw)

gt_filtered = gloria_remove_weak_solvents(gt_raw, min_hbonds=MIN_HBONDS)
water_coords = extract_solvent_coords(gt_filtered)

stripped = gt_raw.remove_solvents()

print(f'PDB: {DEMO_PDB}')
print(f'Real waters with >= {MIN_HBONDS} H-bonds : {len(water_coords)}')
print(f'Non-solvent atoms in stripped structure  : {stripped.atoms["is_present"].sum()}')

### 1b. Build atom info arrays

- `all_atom_info`: every heavy atom in the stripped (water-free) structure → used for Q1 and Q2
- `hbond_cand_info`: N/O/F atoms only → used for Q3

In [ ]:
all_atom_info   = build_all_atom_info(stripped)
hbond_cand_info = build_hbond_candidate_info(stripped)

el_counts = {}
for el in all_atom_info['elements']:
    el_counts[el] = el_counts.get(el, 0) + 1

print(f'Total heavy atoms   : {len(all_atom_info["coords"])}')
print(f'H-bond candidates (N/O/F): {len(hbond_cand_info["coords"])}')
print(f'Element breakdown   : {dict(sorted(el_counts.items(), key=lambda x: -x[1]))}')

### 1c. Q1 — K-nearest atom identities

For each water, retrieve the k=1…K_MAX nearest heavy atoms (by Euclidean distance).
Inspect distances and element labels for the first few waters.

In [ ]:
k_nearest = get_k_nearest_identities(water_coords, all_atom_info, k_max=K_MAX)

print('Shape of distances array :', k_nearest['distances'].shape)  # (n_waters, K_MAX)
print('\nFirst 3 waters — nearest 5 atoms:')
for w in range(min(3, len(water_coords))):
    row = [
        f"{k_nearest['elements'][w,k]}({k_nearest['molcat_labels'][w,k]}) {k_nearest['distances'][w,k]:.2f}Å"
        for k in range(5)
    ]
    print(f'  water {w}: ' + '  |  '.join(row))

### 1d. Q2 — Per-atom-type minimum distances

For each element type (and separately for each molcat_label), compute the per-water minimum distance to any atom of that type.
The global minimum tells us the closest approach ever observed.

In [ ]:
atomtype_dists = compute_atomtype_nearest_dists(water_coords, all_atom_info, label_field='elements')
molcat_dists   = compute_atomtype_nearest_dists(water_coords, all_atom_info, label_field='molcat_labels')

print('=== Element-level global minima ===')
print_atomtype_min_table(atomtype_dists)

print('\n=== Mol-category + backbone/sidechain global minima ===')
print_atomtype_min_table(molcat_dists)

### 1e. Q3 — H-bond partner distances

Collect all contacts (N/O/F atoms within [2.5, 3.5] Å of each water).
Each contact dict records: distance, element, mol_category, atom_name, bb_sc.

In [ ]:
hbond_contacts = compute_hbond_partner_distances(water_coords, hbond_cand_info)

all_dists = [c['distance'] for c in hbond_contacts]
print(f'Total H-bond contacts : {len(hbond_contacts)}')
print(f'Per-water mean        : {len(hbond_contacts) / max(len(water_coords), 1):.2f}')
if all_dists:
    print(f'Distance range        : [{min(all_dists):.3f}, {max(all_dists):.3f}] Å')
    print(f'Median distance       : {np.median(all_dists):.3f} Å')

# Breakdown by partner element
from collections import Counter
el_counts = Counter(c['element'] for c in hbond_contacts)
print(f'\nBy partner element: {dict(el_counts.most_common())}')

---
## Part 2 — Multi-PDB analysis

Run all three analyses across every entry in the PDB list.
Results are stored as a list of per-PDB dicts; aggregation functions pool them.

In [ ]:
results = collect_results(
    pdb_ids,
    min_hbonds = MIN_HBONDS,
    k_max      = K_MAX,
    npz_root   = NPZ_ROOT,
)
print(f'\nSuccessfully processed {len(results)}/{len(pdb_ids)} structures.')
n_waters_total = sum(r['n_waters'] for r in results)
print(f'Total real ≥{MIN_HBONDS}-hbond waters: {n_waters_total}')

In [ ]:
# Pool k-nearest identity arrays across all structures
all_distances    = aggregate_k_nearest(results, 'distances')     # (n_waters_total, K_MAX)
all_elements     = aggregate_k_nearest(results, 'elements')      # (n_waters_total, K_MAX)
all_molcat_labs  = aggregate_k_nearest(results, 'molcat_labels') # (n_waters_total, K_MAX)

# Pool per-atom-type distance distributions
all_atomtype_dists = aggregate_dist_dict(results, 'atomtype_dists')  # {element: array}
all_molcat_dists   = aggregate_dist_dict(results, 'molcat_dists')    # {molcat_label: array}

# Pool H-bond contact dicts
all_hbond_contacts = aggregate_hbond_contacts(results)

print(f'K-nearest distance array shape : {all_distances.shape}')
print(f'Element-level atom types found : {sorted(all_atomtype_dists)}')
print(f'Total H-bond contacts          : {len(all_hbond_contacts)}')

---
## Part 3 — Q1: Identity of k-th nearest atom

Stacked bar charts: for each rank k, what fraction of waters have each atom type as their k-th nearest neighbor?

Two views:
- **By element** (C, N, O, S, P, …): shows the chemical identity
- **By mol-category + backbone/sidechain**: shows the structural context

### 3a. Stacked bar by element

In [ ]:
fig = plot_k_nearest_stacked_bar(
    all_identities       = all_elements,
    all_distances        = all_distances,
    identity_label_type  = 'elements',
    k_max                = K_MAX,
    top_n_categories     = 8,
)
fig.suptitle(f'Q1: Identity of k-th nearest atom — by element\n'
             f'(n = {len(all_distances)} real ≥{MIN_HBONDS}-hbond waters, {len(results)} structures)',
             y=1.03)
plt.show()

### 3b. Stacked bar by mol-category + backbone/sidechain

Shows whether the k-th nearest atom tends to be protein backbone, sidechain, nucleic base, etc.

In [ ]:
fig = plot_k_nearest_stacked_bar(
    all_identities       = all_molcat_labs,
    all_distances        = all_distances,
    identity_label_type  = 'molcat_labels',
    k_max                = K_MAX,
    top_n_categories     = 8,
)
fig.suptitle(f'Q1: Identity of k-th nearest atom — by mol-category + backbone/sidechain\n'
             f'(n = {len(all_distances)} real ≥{MIN_HBONDS}-hbond waters, {len(results)} structures)',
             y=1.03)
plt.show()

### 3c. Distance distributions by rank k

Box plots showing the actual distance to the k-th nearest atom, collapsing identity.
Tells us at what Å range each rank sits.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))

data_by_rank = [
    all_distances[:, k][np.isfinite(all_distances[:, k])]
    for k in range(K_MAX)
]
bp = ax.boxplot(data_by_rank, positions=np.arange(1, K_MAX + 1),
                widths=0.5, patch_artist=True, showfliers=False,
                medianprops=dict(color='black', linewidth=1.5))
for patch in bp['boxes']:
    patch.set_facecolor('#4472C4')
    patch.set_alpha(0.7)

ax.set_xticks(range(1, K_MAX + 1))
ax.set_xticklabels([f'k={k}' for k in range(1, K_MAX + 1)])
ax.set_xlabel('rank k')
ax.set_ylabel('distance (Å)')
ax.set_title(f'Distance to k-th nearest atom (all heavy atoms, n={len(all_distances)} waters)')
fig.tight_layout()
plt.show()

---
## Part 4 — Q2: Per-atom-type minimum distance distributions

For each atom type, the distribution of **per-water minimum distances** — i.e., for each water,
what is the closest any atom of that type ever gets?

The global minimum (dashed vertical line) tells us the closest approach ever observed.

### 4a. Global minimum table — by element

In [ ]:
print('=== Global minimum and median distances to each element type ===')
print_atomtype_min_table(all_atomtype_dists)

### 4b. Global minimum table — by mol-category + backbone/sidechain

In [ ]:
print('=== Global minimum and median distances to each mol-cat + bb/sc label ===')
print_atomtype_min_table(all_molcat_dists)

### 4c. Histograms — by element

One subplot per element.  Sorted by global minimum (closest first).

In [ ]:
fig = plot_atomtype_min_dist_distributions(
    dist_dict  = all_atomtype_dists,
    label_type = 'elements',
    max_dist   = 6.0,
)
plt.show()

### 4d. Histograms — by mol-category + backbone/sidechain

In [ ]:
fig = plot_atomtype_min_dist_distributions(
    dist_dict  = all_molcat_dists,
    label_type = 'molcat_labels',
    max_dist   = 6.0,
)
plt.show()

---
## Part 5 — Q3: H-bond partner distance distributions

For each water, its H-bond partners are the N/O/F atoms within [2.5, 3.5] Å.
We show:
- The overall distance distribution (all contacts combined)
- Breakdown by partner element (N vs O — different H-bond geometry expected)
- Breakdown by mol-category + backbone/sidechain (backbone O vs sidechain O, etc.)

### 5a. Overall distribution

In [ ]:
fig = plot_hbond_distance_histogram(
    contacts  = all_hbond_contacts,
    split_by  = None,
    figsize   = (6, 3.5),
)
plt.show()

### 5b. Split by partner element (N vs O vs F)

In [ ]:
fig = plot_hbond_distance_histogram(
    contacts  = all_hbond_contacts,
    split_by  = 'element',
    figsize   = (6, 3.5),
)
fig.axes[0].set_title(
    f'H-bond partner distances — split by element\n'
    f'(n = {len(all_hbond_contacts)} contacts, {n_waters_total} waters)'
)
plt.show()

### 5c. Split by mol-category + backbone/sidechain

In [ ]:
fig = plot_hbond_distance_histogram(
    contacts  = all_hbond_contacts,
    split_by  = 'molcat_label',
    figsize   = (8, 4),
)
fig.axes[0].set_title(
    f'H-bond partner distances — split by mol-cat + backbone/sidechain\n'
    f'(n = {len(all_hbond_contacts)} contacts)'
)
plt.show()

### 5d. Summary statistics per partner type

In [ ]:
from collections import defaultdict

for split_key in ('element', 'molcat_label'):
    print(f'\n=== H-bond partner distances by {split_key} ===')
    groups = defaultdict(list)
    for c in all_hbond_contacts:
        groups[c[split_key]].append(c['distance'])

    rows = []
    for grp, dists in groups.items():
        d = np.array(dists)
        rows.append((grp, float(d.min()), float(np.median(d)), float(d.mean()), float(d.max()), len(d)))
    rows.sort(key=lambda r: r[1])

    hdr = f'{split_key:<28}  {"min":>6}  {"med":>6}  {"mean":>6}  {"max":>6}  {"n":>7}'
    print(hdr)
    print('-' * len(hdr))
    for grp, mn, med, mean, mx, n in rows:
        print(f'{grp:<28}  {mn:>6.3f}  {med:>6.3f}  {mean:>6.3f}  {mx:>6.3f}  {n:>7}')